# Notebook 1: 데이터 파이프라인
실행 후 Output을 데이터셋으로 저장해서 Notebook 2, 3에서 재사용

In [ ]:
# 1. 코드 복사
import os, shutil

def find_input_dir():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'configs' in dirs and 'src' in dirs:
            return root
    return None

input_dir = find_input_dir()
assert input_dir, 'Triage 데이터셋을 찾을 수 없습니다'
print('Input dir:', input_dir)

dest = '/kaggle/working/Triage'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(input_dir, dest)
os.chdir(dest)
print('Files:', os.listdir('.'))

In [ ]:
# 2. 패키지 설치
!pip install -q omegaconf lifelines optuna statsmodels pingouin pyarrow

In [ ]:
# 3. PhysioNet Challenge 2019 데이터 다운로드
import urllib.request, os, concurrent.futures
from pathlib import Path

BASE_URL = 'https://physionet.org/files/challenge-2019/1.0.0/training'
SETS = {'training_setA': (1, 20336), 'training_setB': (100001, 120000)}

def download_one(args):
    set_name, pid, out_dir = args
    fname = f'p{pid:06d}.psv'
    dest = out_dir / fname
    if dest.exists():
        return False
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{set_name}/{fname}', dest)
        return True
    except:
        return False

for set_name, (start, end) in SETS.items():
    out_dir = Path(f'data/raw/{set_name}')
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = len(list(out_dir.glob('*.psv')))
    total = end - start + 1
    print(f'{set_name}: {existing}/{total}')
    if existing >= total * 0.99:
        print('  Skip'); continue
    tasks = [(set_name, pid, out_dir) for pid in range(start, end+1)
             if not (out_dir/f'p{pid:06d}.psv').exists()]
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
        for i, ok in enumerate(ex.map(download_one, tasks)):
            if (i+1) % 5000 == 0:
                print(f'  {i+1}/{len(tasks)}')
    print(f'{set_name} 완료: {len(list(out_dir.glob("*.psv")))} 파일')

In [ ]:
# 4. 데이터 파이프라인 실행
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'
os.makedirs('results/logs', exist_ok=True)

result = subprocess.run(
    [sys.executable, 'scripts/run_pipeline.py', '--stage', 'data'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 5. 처리된 데이터를 Output에 복사 (다음 노트북에서 재사용)
import shutil
shutil.copytree('/kaggle/working/Triage/data', '/kaggle/working/data')
print('저장 완료')
import os
for split in ['train', 'val', 'test']:
    n = len(list(__import__('pathlib').Path(f'/kaggle/working/data/processed/{split}').glob('*.pt')))
    print(f'{split}: {n} 파일')